In [ ]:
!pip install pdfplumber sentence-transformers langchain-community

In [ ]:
!pip install langchain langchain-text-splitters qdrant-client

In [ ]:
!pip install -U langchain-huggingface

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ── config.py (single source of truth) ───────────────────────────────────────
EMBEDDING_MODEL: str    = "BAAI/bge-large-en-v1.5"
EMBEDDING_DEVICE: str   = "cuda"          # or "cpu"
EMBEDDING_BATCH_SIZE: int = 12
QDRANT_URL: str         = "http://localhost:6333"
QDRANT_API_KEY: str     = ""
QDRANT_COLLECTION: str  = "ai_tutor_docs"
QDRANT_VECTOR_SIZE: int = 1024
CHUNK_SIZE: int         = 512           # fits most embedding model windows
CHUNK_OVERLAP: int      = 100

In [ ]:
import os
os.environ["QDRANT_PATH"] = "/kaggle/working/qdrant_db"

In [ ]:
# ── ingest.py ─────────────────────────────────────────────────────────────────
import json, uuid, logging
import pandas as pd
import pdfplumber, re

from typing import List

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct


logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


# ── STEP 1: JSON ──────────────────────────────────────────────────────────────
def process_json_to_docs(json_file_path: str) -> List[Document]:
    with open(json_file_path) as f:
        data = json.load(f)

    df = pd.DataFrame(data["intents"]).explode("patterns")

    docs = []
    for _, row in df.iterrows():
        if pd.isna(row["patterns"]) or not row["responses"]:
            continue

        # FIX: embed Q+A together so the answer is part of the vector
        all_responses = " | ".join(str(r) for r in row["responses"])
        text = f"Q: {row['patterns'].strip()}\nA: {all_responses}"

        docs.append(Document(
            page_content=text,
            metadata={"tag": str(row["tag"]), "source": "json_intents"},
        ))
    logger.info(f"JSON: loaded {len(docs)} documents.")
    return docs


# ── STEP 2: PDF ───────────────────────────────────────────────────────────────
def process_pdf_to_docs(pdf_file_path: str) -> List[Document]:
    raw_text = ""
    with pdfplumber.open(pdf_file_path) as pdf:
        for page in pdf.pages:
            extracted = page.extract_text()
            if extracted:
                raw_text += extracted + "\n"

    # Collapse soft line-breaks but keep paragraph boundaries
    cleaned = re.sub(r"\n(?!\d+\.)", " ", raw_text)
    pattern = r"(\d+)\.\s+(.*?)\s+A\.\)\s+(.*?)(?=\s+\d+\.|$)"
    matches = re.findall(pattern, cleaned, re.DOTALL)

    docs = []
    for q_num, question, answer in matches:
        # FIX: embed Q+A together
        text = f"Q: {question.strip()}\nA: {answer.strip()}"
        docs.append(Document(
            page_content=text,
            metadata={"question": question.strip(), "source": f"pdf_qna_{q_num}"},
        ))
    logger.info(f"PDF: parsed {len(docs)} Q&A pairs.")
    return docs


# ── STEP 3: CHUNK ─────────────────────────────────────────────────────────────
def split_documents(docs: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    logger.info(f"Split into {len(chunks)} chunks.")
    return chunks


# ── STEP 4: QDRANT SETUP ──────────────────────────────────────────────────────

def get_client() -> QdrantClient:
    # Toggle: local file mode (Kaggle/dev) vs server mode (production)
    qdrant_path = os.getenv("QDRANT_PATH", "")   # set this env var on Kaggle
    if qdrant_path:
        return QdrantClient(path=qdrant_path)
    return QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY or None,
        timeout=60,
    )

def ensure_collection(client: QdrantClient) -> None:
    # Check if it exists
    existing = {c.name for c in client.get_collections().collections}
    
    # If it exists but has the wrong dimension, we must drop it
    if QDRANT_COLLECTION in existing:
        # Check current collection info (optional but safer)
        info = client.get_collection(QDRANT_COLLECTION)
        if info.config.params.vectors.size != QDRANT_VECTOR_SIZE:
            logger.info(f"Dimension mismatch. Deleting existing collection '{QDRANT_COLLECTION}'.")
            client.delete_collection(QDRANT_COLLECTION)
            existing.remove(QDRANT_COLLECTION)

    if QDRANT_COLLECTION not in existing:
        client.create_collection(
            collection_name=QDRANT_COLLECTION,
            vectors_config=VectorParams(size=QDRANT_VECTOR_SIZE, distance=Distance.COSINE),
        )
        logger.info(f"Created collection '{QDRANT_COLLECTION}' with size {QDRANT_VECTOR_SIZE}.")

# ── STEP 5: EMBED + UPSERT ────────────────────────────────────────────────────
def get_embedder() -> HuggingFaceEmbeddings:
    # FIX: pass device from config; add BGE-specific kwargs
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": EMBEDDING_DEVICE},
        encode_kwargs={"normalize_embeddings": True},  # required for BGE cosine similarity
    )

def ingest_chunks(chunks: List[Document]) -> None:
    embedder = get_embedder()
    client   = get_client()
    ensure_collection(client)

    total = len(chunks)
    for start in range(0, total, EMBEDDING_BATCH_SIZE):
        batch = chunks[start : start + EMBEDDING_BATCH_SIZE]
        texts = [c.page_content for c in batch]
        vectors = embedder.embed_documents(texts)

        points = [
            PointStruct(
                id=str(uuid.uuid4()),
                vector=vectors[i],
                payload={
                    "text": texts[i],
                    **batch[i].metadata,
                },
            )
            for i in range(len(batch))
        ]
        client.upsert(collection_name=QDRANT_COLLECTION, points=points)
        logger.info(f"Upserted {min(start + EMBEDDING_BATCH_SIZE, total)}/{total}")

    logger.info("Ingestion complete.")


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main() -> None:
    json_path = "/kaggle/input/datasets/mujtabamatin/computer-science-theory-qa-dataset/intents.json"
    pdf_path  = "/kaggle/input/datasets/nguynvnln22028281/500-data-engineer-qa/500 Data Engineering Interview Questions and Answers.pdf"

    json_docs, pdf_docs = [], []

    try:
        json_docs = process_json_to_docs(json_path)
    except FileNotFoundError:
        logger.error(f"JSON not found: {json_path}")

    try:
        pdf_docs = process_pdf_to_docs(pdf_path)
    except FileNotFoundError:
        logger.error(f"PDF not found: {pdf_path}")

    all_docs = json_docs + pdf_docs
    if not all_docs:
        logger.error("No documents loaded. Exiting.")
        return

    logger.info(f"Total documents: {len(all_docs)}")
    chunks = split_documents(all_docs)
    ingest_chunks(chunks)


if __name__ == "__main__":
    main()

In [ ]:
import shutil

shutil.make_archive(
    base_name="/kaggle/working/qdrant_db_export",  # output zip name
    format="zip",
    root_dir="/kaggle/working",
    base_dir="qdrant_db"                           # folder to zip
)
# Then download qdrant_db_export.zip from Kaggle's output panel